In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# CSV dosyasını Bronze Files klasöründen oku
df = spark.read.option("header", "true").option("inferSchema", "true").csv("Files/musteriler.csv")

# Tablo olarak Bronze katmanındaki Tables klasörüne yaz
df.write.format("delta").mode("overwrite").saveAsTable("musteriler_bronze")

print("Musteriler tablosu başarıyla Bronze katmanına Delta formatında yazıldı!")

StatementMeta(, eb51e671-a356-457b-ba26-6638d6e28228, 3, Finished, Available, Finished, False)

Musteriler tablosu başarıyla Bronze katmanına Delta formatında yazıldı!


In [3]:

# Servis kayıtları dosyasını Delta tablosuna çevir
df_servis = spark.read.option("header", "true").option("inferSchema", "true").csv("Files/servis_kayitlari.csv")
df_servis.write.format("delta").mode("overwrite").saveAsTable("servis_kayitlari_bronze")

print("Tüm Bronze tabloları (musteriler, araclar, servis_kayitlari) başarıyla oluşturuldu!")

StatementMeta(, eb51e671-a356-457b-ba26-6638d6e28228, 5, Finished, Available, Finished, False)

Tüm Bronze tabloları (musteriler, araclar, servis_kayitlari) başarıyla oluşturuldu!


In [1]:
df = spark.sql("SELECT * FROM silver_lakehouse.dbo.musteriler_silver LIMIT 1000")
display(df)

StatementMeta(, 80bc6a67-af9c-40a5-a91f-6d954c3e21c8, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff8092bc-18de-4ab0-8a18-ef00a9866a2d)

In [2]:
df = spark.sql("SELECT * FROM bronze_lakehouse.dbo.musteriler_bronze LIMIT 1000")
display(df)

StatementMeta(, 80bc6a67-af9c-40a5-a91f-6d954c3e21c8, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9da688ba-661c-4b45-ac51-f7e33db631a2)

In [3]:
df = spark.sql("SELECT * FROM bronze_lakehouse.dbo.servis_kayitlari_bronze LIMIT 1000")
display(df)

StatementMeta(, 80bc6a67-af9c-40a5-a91f-6d954c3e21c8, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56878c64-2cac-4d76-944c-b302913c1fd2)

In [4]:
df = spark.sql("SELECT * FROM silver_lakehouse.dbo.servis_kayitlari_silver LIMIT 1000")
display(df)

StatementMeta(, 80bc6a67-af9c-40a5-a91f-6d954c3e21c8, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 00e51722-86ac-43b9-8cf1-8ddfa148eef7)

In [2]:
# Araclar dosyasını Delta tablosuna çevir
df_araclar = spark.read.option("header", "true").option("inferSchema", "true").csv("Files/araclar.csv")
df_araclar.write.format("delta").mode("overwrite").saveAsTable("araclar_bronze")

StatementMeta(, eb51e671-a356-457b-ba26-6638d6e28228, 4, Finished, Available, Finished, False)

In [4]:
from pyspark.sql import functions as F

# Müşteriler verisini Bronze'dan oku, temizle ve Silver'a yaz
df_musteri = spark.read.table("bronze_lakehouse.dbo.musteriler_bronze")
df_musteri_clean = df_musteri.dropDuplicates(["musteri_id"]).filter(F.col("musteri_id").isNotNull())
df_musteri_clean.write.format("delta").mode("overwrite").saveAsTable("silver_lakehouse.dbo.musteriler_silver")

print("Musteriler tablosu Silver katmanına başarıyla aktarıldı!")

StatementMeta(, eb51e671-a356-457b-ba26-6638d6e28228, 6, Finished, Available, Finished, False)

Musteriler tablosu Silver katmanına başarıyla aktarıldı!


In [5]:
# 2) Araçlar verisini oku, temizle ve Silver'a yaz
df_arac = spark.read.table("bronze_lakehouse.dbo.araclar_bronze")
df_arac_clean = df_arac.dropDuplicates(["arac_id"]).filter(F.col("arac_id").isNotNull())
df_arac_clean.write.format("delta").mode("overwrite").saveAsTable("silver_lakehouse.dbo.araclar_silver")

# 3) Servis kayıtları verisini oku, temizle ve Silver'a yaz
df_servis = spark.read.table("bronze_lakehouse.dbo.servis_kayitlari_bronze")
df_servis_clean = df_servis.dropDuplicates(["servis_id"]).filter(F.col("servis_id").isNotNull())
df_servis_clean.write.format("delta").mode("overwrite").saveAsTable("silver_lakehouse.dbo.servis_kayitlari_silver")

print("Tüm Silver tabloları başarıyla oluşturuldu ve temizlendi!")

StatementMeta(, eb51e671-a356-457b-ba26-6638d6e28228, 7, Finished, Available, Finished, False)

Tüm Silver tabloları başarıyla oluşturuldu ve temizlendi!


In [6]:
from pyspark.sql import functions as F

# Silver katmanındaki tabloları oku
df_musteri = spark.read.table("silver_lakehouse.dbo.musteriler_silver")
df_arac = spark.read.table("silver_lakehouse.dbo.araclar_silver")
df_servis = spark.read.table("silver_lakehouse.dbo.servis_kayitlari_silver")

# Tabloları birleştirip iş soruları için özet bir Gold tablo oluşturalım
gold_df = (df_servis
    .join(df_arac, "arac_id")
    .join(df_musteri, "musteri_id")
    .groupBy("yakit_tipi", "sehir")
    .agg(
        F.avg("memnuniyet_puani").alias("ortalama_memnuniyet"),
        F.count("*").alias("toplam_servis_sayisi")
    )
)

# Gold Lakehouse Tables altına yaz
gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_lakehouse.dbo.yakit_memnuniyet_gold")

print("Gold katmanı tablosu başarıyla oluşturuldu!")

StatementMeta(, eb51e671-a356-457b-ba26-6638d6e28228, 8, Finished, Available, Finished, False)

Gold katmanı tablosu başarıyla oluşturuldu!
